# Phase 9A — Twisted Dirac / H_T Finite-Basis Scaffold

This notebook moves beyond the scalar Berger proxy by constructing a finite Dirac proxy basis with chirality, Hopf charge, sectors, twist shifts, and protected zero modes.

Limitation: this is `DIRAC_PROXY_LEVEL_1`, not the final analytic twisted Dirac or full `H_T` spectrum. The no-extra-light-state theorem remains open.

In [ ]:
from pathlib import Path
import sys
import numpy as np

root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root / 'src'))

from ht_operator import (
    alpha_scaled_a,
    build_ht_spectrum,
    default_twist_parameter_grid,
    gap_report,
    scan_twisted_dirac_robustness,
)
from positivity import psd_barrier_from_q
from spectral_gap import MU_H, natural_lambda2
from twisted_dirac import (
    DIRAC_PROXY_LEVEL,
    build_dirac_basis,
    complement_spectrum,
    dirac_squared_spectrum,
    zero_mode_subspace,
)

MU_H, natural_lambda2(), DIRAC_PROXY_LEVEL

## Finite Dirac Basis and Twist Parameters

In [ ]:
k_max = 4
basis = build_dirac_basis(k_max=k_max)
twist_params = {
    'dirac_scale': 2.0,
    'boundary_strength': 0.05,
    'chirality_shift': 0.01,
    'hopf_shift': 0.0,
    'sector_shifts': {},
}
spectrum = dirac_squared_spectrum(basis, a=1.0, twist_params=twist_params)
{'basis_size': len(basis), 'sample_mode': basis[0], 'twist_params': twist_params}

## Protected Zero-Mode Subspace and Complement Spectrum

In [ ]:
zero_modes = zero_mode_subspace(basis, index_count=3)
complement = complement_spectrum(spectrum, zero_modes)
{
    'protected_zero_modes': zero_modes.shape[1],
    'first_complement_dirac_squared': complement[0]['eigenvalue'],
    'first_complement_mode': complement[0]['mode'],
}

## H_T Construction and Pass/Fail Against mu_H

In [ ]:
ht_spectrum = build_ht_spectrum(spectrum, lambda2=natural_lambda2(), mu_h=MU_H)
gap_report(spectrum, zero_modes, lambda2=natural_lambda2(), mu_h=MU_H)

## PSD Profile Contribution

In [ ]:
profile = psd_barrier_from_q(np.eye(len(spectrum)))
gap_report(spectrum, zero_modes, lambda2=natural_lambda2(), mu_h=MU_H, profile_term=profile)

## Sensitivity to Twist Parameters

In [ ]:
sensitivity_rows = []
for scale in [1.0, 1.5, 2.0, 3.0]:
    params = dict(twist_params, dirac_scale=scale)
    spec = dirac_squared_spectrum(basis, a=1.0, twist_params=params)
    report = gap_report(spec, zero_modes, lambda2=natural_lambda2(), mu_h=MU_H)
    sensitivity_rows.append({
        'dirac_scale': scale,
        'first_complement_dirac_squared': report['first_complement_dirac_squared'],
        'first_ht_complement_gap': report['first_ht_complement_gap'],
        'passes_mu_h': report['passes_mu_h'],
    })
sensitivity_rows

## Phase 9B — Robustness Audit

The following scans stress-test `DIRAC_PROXY_LEVEL_1` across basis size, Berger anisotropy, sector choices, and nearby twist perturbations. These are robustness checks only.

In [ ]:
k_a_rows = scan_twisted_dirac_robustness(
    k_max_values=[4, 6, 8, 10],
    a_values=[0.573, 1.0, alpha_scaled_a()],
    twist_param_grid=default_twist_parameter_grid()[:1],
    sectors=[('lepton', 'up', 'down')],
)
k_a_rows

In [ ]:
twist_rows = scan_twisted_dirac_robustness(
    k_max_values=[4, 6, 8, 10],
    a_values=[1.0],
    twist_param_grid=default_twist_parameter_grid(),
    sectors=[('lepton', 'up', 'down')],
)
twist_rows

In [ ]:
sector_rows = scan_twisted_dirac_robustness(
    k_max_values=[4],
    a_values=[1.0],
    twist_param_grid=default_twist_parameter_grid()[:1],
    sectors=[('lepton', 'up', 'down'), ('lepton',), ('up',), ('down',)],
)
sector_rows

In [ ]:
all_rows = k_a_rows + twist_rows + sector_rows
worst_row = min(all_rows, key=lambda row: row['margin'])
best_row = max(all_rows, key=lambda row: row['margin'])
{'best_row': best_row, 'worst_row': worst_row, 'failures': [row for row in all_rows if not row['passes']]}

This finite-basis construction is an audit scaffold only. It must be replaced by the full twisted Dirac `H_T` spectrum before any no-extra-light-state theorem can be claimed.